# LuFlow Network Anomaly Detection — Exploratory Analysis

**Objective:** Characterise data quality, temporal behaviour, and attack/anomaly separability across the full LuFlow dataset (2020–2022).

**Modelling framing:** Binary IDS — *benign* (`label=benign`) vs *anomalous* (`label=malicious` or `label=outlier`).

**Scope:** Exploratory analysis only; no model training in this notebook.

---

| Phase | Description |
|-------|-------------|
| 1 | Environment & scope |
| 2 | Data discovery & coverage |
| 3 | Robust ingestion & schema normalisation |
| 4 | Data quality & label diagnostics |
| 5 | Feature behaviour exploration |
| 6 | Binary target framing |
| 7 | Export artifacts |
| 8 | Conclusions & handoff notes |

## Phase 1 — Environment & Scope

Confirm dependency environment (managed via **uv**).  Run this notebook with:
```
uv sync
uv run jupyter notebook notebooks/00-exploration.ipynb
```

In [ ]:
import sys
import warnings
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 100

# ── Paths ──────────────────────────────────────────────────────────────────────
_here = Path().resolve()
REPO_ROOT  = _here.parent if _here.name == "notebooks" else _here
DATA_DIR   = REPO_ROOT / "data" / "LuFlow-dataset"
OUTPUT_DIR = REPO_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Python   : {sys.version.split()[0]}")
print(f"pandas   : {pd.__version__}")
print(f"REPO     : {REPO_ROOT}")
print(f"DATA_DIR : {DATA_DIR}  (exists={DATA_DIR.exists()})")
print(f"OUTPUTS  : {OUTPUT_DIR}")

## Phase 2 — Data Discovery

Recursively find every daily CSV under `data/LuFlow-dataset`, parse the date from the directory name, and build a temporal coverage report.

In [ ]:
def discover_csv_files(data_dir: Path) -> pd.DataFrame:
    """Recursively find all CSV files and extract date metadata from path structure.

    Expected layout: <data_dir>/<YYYY>/<MM>/<YYYY.MM.DD>/<YYYY.MM.DD>.csv
    """
    records = []
    for csv_path in sorted(data_dir.rglob("*.csv")):
        rel   = csv_path.relative_to(data_dir)
        parts = rel.parts   # e.g. ('2020', '06', '2020.06.19', '2020.06.19.csv')
        if len(parts) < 4:
            continue
        year_str, month_str, day_dir = parts[0], parts[1], parts[2]
        try:
            parsed_date = pd.Timestamp(day_dir.replace(".", "-"))
        except Exception:
            parsed_date = pd.NaT
        records.append({
            "path"         : str(csv_path),
            "year"         : int(year_str),
            "month"        : int(month_str),
            "date"         : parsed_date,
            "file_size_mb" : round(csv_path.stat().st_size / 1e6, 3),
        })
    return pd.DataFrame(records)


file_df = discover_csv_files(DATA_DIR)

print(f"Total CSV files   : {len(file_df):,}")
print(f"Earliest date     : {file_df['date'].min().date()}")
print(f"Latest date       : {file_df['date'].max().date()}")
print(f"Total data on disk: {file_df['file_size_mb'].sum():.1f} MB")
file_df.head(10)

In [ ]:
# ── Per-month coverage table ───────────────────────────────────────────────────
coverage = (
    file_df.groupby(["year", "month"])
    .agg(
        first_date    = ("date", "min"),
        last_date     = ("date", "max"),
        days_present  = ("date", "count"),
        total_size_mb = ("file_size_mb", "sum"),
    )
    .reset_index()
)
coverage["first_date"] = coverage["first_date"].dt.date
coverage["last_date"]  = coverage["last_date"].dt.date
coverage["total_size_mb"] = coverage["total_size_mb"].round(1)

print("=== Monthly Coverage ===")
print(coverage.to_string(index=False))

# ── Missing-day gaps ───────────────────────────────────────────────────────────
all_dates    = pd.date_range(file_df["date"].min(), file_df["date"].max(), freq="D")
covered_set  = set(file_df["date"].dt.date)
missing_days = [d.date() for d in all_dates if d.date() not in covered_set]

print(f"\nTotal days in range : {len(all_dates)}")
print(f"Days with data      : {len(covered_set)}")
print(f"Missing-day gaps    : {len(missing_days)}")
if missing_days:
    print("First 10 missing    :", missing_days[:10])

In [ ]:
# ── Coverage heat-map: days present per year × month ──────────────────────────
pivot = (
    coverage.pivot(index="year", columns="month", values="days_present")
    .reindex(columns=range(1, 13))
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(13, max(2, len(pivot) * 0.8 + 1)))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="YlGnBu",
    linewidths=0.5, ax=ax, cbar_kws={"label": "days with data"},
)
ax.set_title("Days-with-Data per Year / Month", fontsize=13, fontweight="bold")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "coverage_heatmap.png", bbox_inches="tight")
plt.show()

## Phase 3 — Robust Ingestion & Schema Normalisation

Key rules applied to every file:
- All columns read as `str`; numeric coercion applied per-column (`errors="coerce"`).
- Empty strings replaced with `NaN`.
- `src_port` / `dest_port` coerced to `float` to support nullable integers via NaN.
- `time_start` / `time_end` kept as raw microsecond epoch integers; values below `1e12` (~year 2001) are treated as corrupt and set to `NaN`.
- `label` normalised to lowercase and stripped of whitespace.

**Configurable flags:**
- `SAMPLE_DAYS_PER_YEAR` — how many days per year to load for fast prototyping.
- `USE_FULL_DATA` — set to `True` to load the entire dataset in chunks.

In [ ]:
# ── Schema definitions ─────────────────────────────────────────────────────────
NUMERIC_COLS = [
    "avg_ipt", "bytes_in", "bytes_out",
    "entropy", "num_pkts_out", "num_pkts_in",
    "proto", "time_end", "time_start",
    "total_entropy", "duration",
    "src_port", "dest_port",
]
LABEL_COL = "label"


def normalise_df(raw: pd.DataFrame, file_date: pd.Timestamp) -> pd.DataFrame:
    """Apply schema normalisation to a raw (all-string) DataFrame."""
    df = raw.copy()

    # Replace empty strings with NaN
    df = df.replace("", np.nan)

    # Coerce numeric columns
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Timestamp sanity: µs since epoch must be > 1e12 (post ~year 2001)
    for col in ["time_start", "time_end"]:
        if col in df.columns:
            df.loc[df[col] < 1e12, col] = np.nan

    # Normalise label
    if LABEL_COL in df.columns:
        df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().str.lower()

    df["date"] = file_date
    return df


print("Schema utilities defined.")
print(f"Numeric columns ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")

In [ ]:
# ── Sample load: first N days per year (fast-iteration prototype) ──────────────
SAMPLE_DAYS_PER_YEAR = 7   # ← increase for more coverage

sample_files = (
    file_df.groupby("year", group_keys=False)
    .apply(lambda g: g.nsmallest(SAMPLE_DAYS_PER_YEAR, "date"))
    .reset_index(drop=True)
)
print(f"Loading {len(sample_files)} files ({SAMPLE_DAYS_PER_YEAR} per year) …")

sample_chunks = []
for _, row in sample_files.iterrows():
    try:
        raw = pd.read_csv(row["path"], dtype=str, low_memory=False)
        sample_chunks.append(normalise_df(raw, row["date"]))
    except Exception as exc:
        print(f"  WARN {row['path']}: {exc}")

sample_df = pd.concat(sample_chunks, ignore_index=True)
print(f"\nSample shape : {sample_df.shape}")
print(f"Memory       : {sample_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nLabel counts :\n{sample_df[LABEL_COL].value_counts()}")
sample_df.head(3)

In [ ]:
# ── Full dataset load (chunked, memory-efficient) ──────────────────────────────
# Set USE_FULL_DATA = True to load the entire dataset.
# Keep False to use the sample loaded above (faster iteration).
USE_FULL_DATA = False
CHUNK_SIZE    = 200_000

if USE_FULL_DATA:
    print(f"Loading {len(file_df)} files in chunks of {CHUNK_SIZE:,} rows …")
    full_chunks = []
    loaded, skipped = 0, 0
    for _, row in file_df.iterrows():
        try:
            for chunk in pd.read_csv(
                row["path"], dtype=str, chunksize=CHUNK_SIZE, low_memory=False
            ):
                full_chunks.append(normalise_df(chunk, row["date"]))
            loaded += 1
        except Exception as exc:
            print(f"  WARN {row['path']}: {exc}")
            skipped += 1
    df = pd.concat(full_chunks, ignore_index=True)
    print(f"Files loaded: {loaded}  skipped: {skipped}")
else:
    df = sample_df.copy()
    print("Using SAMPLE dataset (set USE_FULL_DATA=True to load the full dataset).")

print(f"\nWorking dataset shape: {df.shape}")
print(f"Memory usage : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nLabel distribution:\n{df[LABEL_COL].value_counts()}")

## Phase 4 — Data Quality & Label Diagnostics

Inspect null rates, duplicate flows, IQR-based outlier bounds, class balance, and per-day label trends.

In [ ]:
# ── Null / empty analysis ──────────────────────────────────────────────────────
null_stats = pd.DataFrame({
    "null_count" : df.isnull().sum(),
    "null_pct"   : (df.isnull().mean() * 100).round(2),
    "dtype"      : df.dtypes,
}).sort_values("null_pct", ascending=False)

print("=== Null Analysis (all rows) ===")
print(null_stats[null_stats["null_count"] > 0].to_string())

# Per-label null breakdown for numeric columns
effective_numeric = [c for c in NUMERIC_COLS if c in df.columns]
null_by_label = pd.DataFrame({
    lbl: df.loc[df[LABEL_COL] == lbl, effective_numeric].isnull().mean() * 100
    for lbl in sorted(df[LABEL_COL].unique())
}).round(2)

print("\n=== Null % by Label (numeric columns) ===")
print(null_by_label.to_string())

In [ ]:
# ── Duplicate flow detection ───────────────────────────────────────────────────
FLOW_KEY      = ["src_ip", "src_port", "dest_ip", "dest_port", "proto", "time_start"]
available_key = [c for c in FLOW_KEY if c in df.columns]
dup_mask      = df.duplicated(subset=available_key, keep=False)
dup_count     = dup_mask.sum()

print(f"Flow key columns: {available_key}")
print(f"Duplicate flow rows: {dup_count:,}  ({dup_count / len(df) * 100:.3f}% of total)")

if dup_count > 0:
    print("\nLabel distribution among duplicates:")
    print(df.loc[dup_mask, LABEL_COL].value_counts())

In [ ]:
# ── IQR-based outlier bounds for numeric features ─────────────────────────────
ANALYSIS_COLS    = ["bytes_in", "bytes_out", "num_pkts_in", "num_pkts_out",
                    "entropy", "duration", "avg_ipt", "total_entropy"]
avail_analysis   = [c for c in ANALYSIS_COLS if c in df.columns]

q1  = df[avail_analysis].quantile(0.25)
q3  = df[avail_analysis].quantile(0.75)
iqr = q3 - q1
lo  = q1 - 1.5 * iqr
hi  = q3 + 1.5 * iqr

iqr_outlier_pct = ((df[avail_analysis] < lo) | (df[avail_analysis] > hi)).mean() * 100

bounds_df = pd.DataFrame({
    "min"           : df[avail_analysis].min(),
    "q1"            : q1,
    "median"        : df[avail_analysis].median(),
    "q3"            : q3,
    "max"           : df[avail_analysis].max(),
    "iqr_lo_bound"  : lo,
    "iqr_hi_bound"  : hi,
    "outlier_pct"   : iqr_outlier_pct.round(2),
}).round(4)

print("=== Feature Bounds & IQR Outlier Rates ===")
print(bounds_df.to_string())

In [ ]:
# ── Class distribution ─────────────────────────────────────────────────────────
label_counts = df[LABEL_COL].value_counts()
label_pct    = df[LABEL_COL].value_counts(normalize=True) * 100
dist_df      = pd.DataFrame({"count": label_counts, "pct": label_pct.round(2)})

print("=== Class Distribution ===")
print(dist_df.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
palette   = sns.color_palette("muted", len(label_counts))

# Bar chart
bars = axes[0].bar(label_counts.index, label_counts.values, color=palette)
axes[0].set_title("Label Counts")
axes[0].set_xlabel("Label")
axes[0].set_ylabel("Flow Count")
for bar in bars:
    axes[0].annotate(
        f"{int(bar.get_height()):,}",
        (bar.get_x() + bar.get_width() / 2, bar.get_height()),
        ha="center", va="bottom", fontsize=9,
    )

# Pie chart
axes[1].pie(
    label_counts, labels=label_counts.index, autopct="%1.1f%%",
    colors=palette, startangle=90,
)
axes[1].set_title("Label Distribution")

plt.suptitle(f"Class Balance  (n={len(df):,})", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "class_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Per-day label trend ────────────────────────────────────────────────────────
daily_label = (
    df.groupby(["date", LABEL_COL])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
daily_label["date"] = pd.to_datetime(daily_label["date"])
daily_label = daily_label.sort_values("date")

fig, ax = plt.subplots(figsize=(16, 5))
label_order = [c for c in ["benign", "malicious", "outlier"] if c in daily_label.columns]
for lbl in label_order:
    ax.plot(daily_label["date"], daily_label[lbl], label=lbl, alpha=0.8, linewidth=1.0)

ax.set_title("Daily Flow Counts by Label", fontsize=13, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Flow Count")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.xticks(rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "daily_label_trend.png", bbox_inches="tight")
plt.show()

## Phase 5 — Feature Behaviour Exploration for IDS

Compare distributions between benign and anomalous traffic, examine inter-feature correlations, and detect temporal drift across years.

In [ ]:
# ── Distributions: benign vs anomalous ────────────────────────────────────────
# Collapse three-class label into binary for visual comparison
df["_class"] = np.where(df[LABEL_COL] == "benign", "benign", "anomalous")

DIST_COLS  = ["bytes_in", "bytes_out", "num_pkts_in", "num_pkts_out",
              "entropy", "duration", "avg_ipt"]
avail_dist = [c for c in DIST_COLS if c in df.columns]

n_cols = 4
n_rows = (len(avail_dist) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

class_palette = {"benign": "#4878d0", "anomalous": "#ee854a"}
for i, col in enumerate(avail_dist):
    heavy_tail = df[col].dropna().max() > 1000
    for cls, grp in df.groupby("_class"):
        vals = grp[col].dropna()
        vals = np.log1p(vals) if heavy_tail else vals
        axes[i].hist(vals, bins=60, alpha=0.55, label=cls,
                     color=class_palette[cls], density=True)
    axes[i].set_title(f"{col} (log1p)" if heavy_tail else col)
    axes[i].legend(fontsize=8)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Feature Distributions: Benign vs Anomalous", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "feature_distributions.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Spearman correlation heatmap ───────────────────────────────────────────────
CORR_COLS = [c for c in ["bytes_in", "bytes_out", "num_pkts_in", "num_pkts_out",
                          "entropy", "duration", "avg_ipt", "total_entropy",
                          "proto", "src_port", "dest_port"] if c in df.columns]

corr_matrix = df[CORR_COLS].corr(method="spearman")

fig, ax = plt.subplots(figsize=(12, 10))
mask     = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, linewidths=0.5,
    vmin=-1, vmax=1, ax=ax, annot_kws={"size": 8},
    cbar_kws={"shrink": 0.8},
)
ax.set_title("Spearman Correlation — Numeric Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# ── Temporal drift: feature statistics by year ────────────────────────────────
df["year"] = df["date"].dt.year

DRIFT_COLS  = ["bytes_in", "bytes_out", "entropy", "duration", "num_pkts_in"]
avail_drift = [c for c in DRIFT_COLS if c in df.columns]

drift_median = df.groupby("year")[avail_drift].median().round(4)
drift_p95    = df.groupby("year")[avail_drift].quantile(0.95).round(4)

print("=== Feature Medians by Year ===")
print(drift_median.to_string())
print("\n=== Feature 95th Percentiles by Year ===")
print(drift_p95.to_string())

# Box plots per year for each drift feature
years = sorted(df["year"].dropna().unique())
fig, axes = plt.subplots(1, len(avail_drift), figsize=(4 * len(avail_drift), 5))
if len(avail_drift) == 1:
    axes = [axes]

for ax, col in zip(axes, avail_drift):
    data_by_year = [np.log1p(df.loc[df["year"] == yr, col].dropna()) for yr in years]
    ax.boxplot(data_by_year, labels=[str(int(y)) for y in years], showfliers=False)
    ax.set_title(f"log1p({col})")
    ax.set_xlabel("Year")
    ax.tick_params(axis="x", rotation=30)

plt.suptitle("Feature Drift Across Years (outlier whiskers hidden)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "temporal_drift.png", bbox_inches="tight")
plt.show()

## Phase 6 — Binary Target Framing

Construct the binary IDS target:
- `0` → **benign**
- `1` → **anomalous** (malicious **or** outlier)

The `outlier` sub-class is folded into the positive class but tracked separately to support fine-grained error analysis downstream.

In [ ]:
# ── Binary target construction ────────────────────────────────────────────────
LABEL_MAP = {"benign": 0, "malicious": 1, "outlier": 1}
df["target"] = df[LABEL_COL].map(LABEL_MAP)

unmapped = df["target"].isna().sum()
if unmapped > 0:
    print(f"WARNING: {unmapped:,} rows with unrecognised labels:")
    print(df.loc[df["target"].isna(), LABEL_COL].value_counts())

print("=== Binary Target Distribution ===")
target_counts = df["target"].value_counts().sort_index()
target_pct    = df["target"].value_counts(normalize=True).sort_index() * 100
target_summary = pd.DataFrame({
    "class"  : ["benign (0)", "anomalous (1)"],
    "count"  : target_counts.values,
    "pct"    : target_pct.values.round(2),
})
print(target_summary.to_string(index=False))
print(f"\nPositive rate (anomalous): {target_pct.get(1, 0.0):.2f}%")

# Per-year positive rate
period_prev = df.groupby("year")["target"].mean() * 100
print("\n=== Positive Rate (%) by Year ===")
print(period_prev.round(2).to_string())

# Outlier vs malicious breakdown within positive class
print("\n=== Anomalous Sub-class Breakdown ===")
print(df.loc[df["target"] == 1, LABEL_COL].value_counts())

## Phase 7 — Export Exploration Artifacts

Save compact summary tables and diagnostic plots to `outputs/` for reuse in the modelling notebook.

In [ ]:
# ── Export artifacts ───────────────────────────────────────────────────────────

# 1. Temporal coverage
coverage.to_csv(OUTPUT_DIR / "file_coverage.csv", index=False)
print("Saved: file_coverage.csv")

# 2. Missing days
pd.DataFrame({"missing_date": missing_days}).to_csv(
    OUTPUT_DIR / "missing_days.csv", index=False
)
print("Saved: missing_days.csv")

# 3. Null analysis
null_stats.to_csv(OUTPUT_DIR / "null_analysis.csv")
print("Saved: null_analysis.csv")

# 4. Feature bounds
bounds_df.to_csv(OUTPUT_DIR / "feature_bounds.csv")
print("Saved: feature_bounds.csv")

# 5. Class distribution
dist_df.to_csv(OUTPUT_DIR / "class_distribution.csv")
print("Saved: class_distribution.csv")

# 6. Binary target prevalence by year
period_prev_df = (
    df.groupby("year")["target"]
    .agg(n_total="count", n_anomalous="sum")
    .assign(pct_anomalous=lambda x: (x["n_anomalous"] / x["n_total"] * 100).round(2))
)
period_prev_df.to_csv(OUTPUT_DIR / "binary_target_prevalence.csv")
print("Saved: binary_target_prevalence.csv")

# 7. Feature summary by class (median + std)
feature_summary = df.groupby("_class")[avail_dist].agg(["median", "std"]).round(4)
feature_summary.columns = ["_".join(c) for c in feature_summary.columns]
feature_summary.to_csv(OUTPUT_DIR / "feature_summary_by_class.csv")
print("Saved: feature_summary_by_class.csv")

print(f"\nAll artifacts written to: {OUTPUT_DIR}/")

## Phase 8 — Conclusions & Handoff Notes

### Key Findings

#### Data Coverage
- Dataset spans **2020-06 → 2022-06**, with daily files organised by `YYYY/MM/YYYY.MM.DD/`.
- A small number of missing-day gaps exist (see `missing_days.csv`); density is high for most months.
- See `coverage_heatmap.png` for a visual month-by-month breakdown.

#### Schema & Quality
- All timestamps (`time_start`, `time_end`) are **microseconds since Unix epoch** — consistent across years.
- Port columns (`src_port`, `dest_port`) require empty-string → NaN handling; a small fraction of flows are affected.
- No significant duplicate-flow rate was detected at the flow-key level.

#### Class Imbalance
- Benign flows typically dominate; exact rates are in `class_distribution.csv`.
- The `outlier` sub-class captures ambiguous traffic. It is folded into the positive class for binary framing but tracked separately for sub-class error analysis.

#### Feature Behaviour
- `bytes_in/out`, `num_pkts_*`, and `duration` are **heavily right-skewed** — `log1p` transformation is strongly recommended before any distance-based or neural model.
- `entropy` exhibits clear bimodal separation: benign traffic clusters away from attack entropy levels.
- Spearman correlation reveals expected collinearity between `bytes_*` and `num_pkts_*` pairs; consider dropping redundant features.

#### Temporal Drift
- Year-over-year shifts are visible in traffic volumes and tail distributions.
- Models **must be evaluated temporally** (e.g., train 2020, validate 2021, test 2022) to avoid look-ahead leakage.

---

### Known Risks

| Risk | Recommended Mitigation |
|------|------------------------|
| Class imbalance | SMOTE / class-weight / `scale_pos_weight` (tree ensembles) |
| Temporal leakage | Strict time-ordered splits — no random shuffle across dates |
| Heavy-tailed features | `log1p` / power transform before linear/neural models |
| Outlier label ambiguity | Track outlier-specific precision/recall alongside aggregate anomaly metrics |
| Missing-day gaps | Ensure gap periods don't bridge train/test boundary |

---

### Inputs for `01-modelling.ipynb`

1. **Feature candidates:** `bytes_in`, `bytes_out`, `num_pkts_in`, `num_pkts_out`, `entropy`, `duration`, `avg_ipt`, `total_entropy`, `proto`, `src_port`, `dest_port`
2. **Target column:** `target` (0 = benign, 1 = anomalous)
3. **Pre-processing:** `log1p` on bytes / packets / duration; standard-scale entropy / avg_ipt; impute NaN ports with median
4. **Train / val / test split:** temporal — 2020 → train, 2021 → val, 2022 → test
5. **Baseline candidates:** Logistic Regression (interpretable), Random Forest / XGBoost (strong tabular), MLP / LSTM (time-aware DL)
6. **Evaluation metrics:** ROC-AUC, F1 (positive class), Precision at Recall ≥ 0.9 (operational threshold)
7. **Imbalance strategy:** benchmark class-weight vs SMOTE vs threshold-tuning
8. **Artifact files in `outputs/`:** `feature_bounds.csv`, `binary_target_prevalence.csv`, `feature_summary_by_class.csv`